# NEXUS / GAIA — Multi-Account Drive ↔ Colab Bridge\nEvidence-first bridge for the shared GAIA/NEXUS Colab corpus. It authenticates the active Colab Google identity, probes canonical Drive file IDs, pulls selected artifacts into `/content/nexus_bridge`, and can push an explicitly selected local file back to the same Drive object. No purge or destructive notebook is auto-executed.\n

In [ ]:
from google.colab import auth\nauth.authenticate_user()\nimport google.auth\nfrom googleapiclient.discovery import build\ncreds, _ = google.auth.default()\ndrive = build('drive', 'v3', credentials=creds, cache_discovery=False)\nprint('Drive API authenticated for the active Colab identity')\n

In [ ]:
REGISTRY = {\n  'great_assimilation_cloud_purge': {'id':'1NOd-M9EtzLh8puQq2S98UPK3avEUZCyH','name':'Great_Assimilation_Cloud_Purge_v2.ipynb','mode':'artifact_only'},\n  'nexus_generation': {'id':'1HQuv92qVIiWdyvVRdYh5H07AjHqWtixi','name':'The_Nexus_Generation.ipynb','mode':'artifact_only'},\n  'gemini_backend': {'id':'1mJPDWyUtXSIxoWHLQWc8TyYEZJVmNp2j','name':'Nexus_Gemini_Colab_Backend.ipynb','mode':'runtime_candidate'},\n  'coherence_record': {'id':'1hnXsw4WXZdpAsLbR4DmwX5-spJidmOpV','name':'quantum_coherence_record.json','mode':'artifact_only'},\n  'coherence_expand': {'id':'1DFWBAgRPnJIbngEO579GoxaoANomb8A8','name':'quantum_coherence_expand.sh','mode':'artifact_only'},\n  'coherence_cron': {'id':'1KpVit_u5q-3dzAsd_ASa8qnT7GCvnPRY','name':'quantum_coherence_cron.sh','mode':'artifact_only'},\n}\nREGISTRY\n

In [ ]:
from pathlib import Path\nimport hashlib, json, io, time\nfrom googleapiclient.http import MediaIoBaseDownload, MediaFileUpload\n\nROOT = Path('/content/nexus_bridge')\nROOT.mkdir(parents=True, exist_ok=True)\nRECEIPTS = ROOT / 'receipts.jsonl'\n\ndef sha256_file(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)\n    return h.hexdigest()\n\ndef receipt(event, **data):\n    row={'ts':time.time(),'event':event,**data}\n    with RECEIPTS.open('a',encoding='utf-8') as f: f.write(json.dumps(row,sort_keys=True)+'\\n')\n    return row\n\ndef probe(key):\n    item=REGISTRY[key]\n    meta=drive.files().get(fileId=item['id'],fields='id,name,mimeType,modifiedTime,size,capabilities(canEdit,canDownload)').execute()\n    receipt('probe',key=key,file_id=item['id'],name=meta.get('name'),can_edit=meta.get('capabilities',{}).get('canEdit'))\n    return meta\n\ndef pull(key):\n    item=REGISTRY[key]\n    out=ROOT/item['name']\n    request=drive.files().get_media(fileId=item['id'])\n    with io.FileIO(out,'wb') as fh:\n        dl=MediaIoBaseDownload(fh,request)\n        done=False\n        while not done: _,done=dl.next_chunk()\n    digest=sha256_file(out)\n    receipt('pull',key=key,file_id=item['id'],path=str(out),sha256=digest,size=out.stat().st_size)\n    return out,digest\n\ndef push(key, local_path):\n    item=REGISTRY[key]\n    local_path=Path(local_path)\n    before=drive.files().get(fileId=item['id'],fields='id,name,modifiedTime,capabilities(canEdit)').execute()\n    if not before.get('capabilities',{}).get('canEdit'):\n        raise PermissionError(f\"Active identity cannot edit {item['id']}\")\n    media=MediaFileUpload(str(local_path),resumable=True)\n    after=drive.files().update(fileId=item['id'],media_body=media,fields='id,name,modifiedTime').execute()\n    digest=sha256_file(local_path)\n    receipt('push',key=key,file_id=item['id'],path=str(local_path),sha256=digest,modifiedTime=after.get('modifiedTime'))\n    return after,digest\n

In [ ]:
# Access probe: the canonical bridge files are shared with the connected Drive identities.\nfor key in REGISTRY:\n    try:\n        m=probe(key)\n        print(key, 'OK', m.get('name'), 'editable=', m.get('capabilities',{}).get('canEdit'))\n    except Exception as e:\n        print(key, 'UNAVAILABLE', type(e).__name__, str(e)[:180])\n

In [ ]:
# Pull the canonical corpus into the current Colab runtime. This does NOT execute any imported notebook or shell script.\nfor key in REGISTRY:\n    try:\n        p,h=pull(key)\n        print(key, p, h[:16])\n    except Exception as e:\n        print(key, 'PULL_FAILED', type(e).__name__, str(e)[:180])\n

## Bidirectional write path\nUse `push('registry_key', '/content/nexus_bridge/file')` only after intentionally editing a local artifact. The bridge records SHA-256 and Drive modification time in `receipts.jsonl`. Imported purge/expansion scripts remain data until separately reviewed and explicitly executed.\n